# A_4_0 — Extraction of promoter justifications

**Input:** `Results/07.xlsx`
**Output:** `Results/Delay_reason_onlyDelayed.xlsx` (n = 484), `Results/Delay_reason_onlyRescheduled.xlsx` (n = 317)

Reshapes the progress fields into long format, one row per investment and
reporting cycle, and isolates the statements flagged as delayed or rescheduled.
The two groups are analysed separately in the following notebooks because they
describe different events: late-stage overruns and early-stage timeline
revisions.

In [10]:
import numpy as np
import pandas as pd

In [11]:
df = pd.read_excel("Results/07.xlsx", header=[0,1], index_col=0)

In [12]:
columns_of_interest = [
    col for col in df.columns 
    if col[1] in ('Inv_index', 'Inv_Progress', 'Inv_Progress Driver','Inv_Status')
]

df_d = df[columns_of_interest].copy()

In [13]:
years_to_remove = ['2012', '2013', '2026','meta']

filtered_columns = [col for col in df_d.columns if col[0] not in years_to_remove]

df_d_filtered = df_d[filtered_columns].copy()

In [14]:
years = sorted({col[0] for col in df_d_filtered.columns if col[0] != 'meta'})

frames = []

for year in years:
    selected_cols = [(year, 'Inv_index'), (year, 'Inv_Progress'), (year, 'Inv_Progress Driver'),(year, 'Inv_Status')]
    selected_cols = [col for col in selected_cols if col in df_d_filtered.columns]
    if selected_cols:
        temp = df_d_filtered[selected_cols].copy()
        temp.columns = [c[1] for c in temp.columns] 
        temp['year'] = year
        temp = temp.reset_index()
        frames.append(temp)

Delay_reason = pd.concat(frames, ignore_index=True)

In [15]:
Delay_reason_clean = Delay_reason[Delay_reason['Inv_index'].notna() & (Delay_reason['Inv_index'] != '')]

Delay_reason_clean = Delay_reason_clean.reset_index(drop=True)

In [16]:
delayed = Delay_reason_clean[Delay_reason_clean['Inv_Progress'].str.contains('Delayed', na=False)]

rescheduled = Delay_reason_clean[Delay_reason_clean['Inv_Progress'].str.contains('Rescheduled', na=False)]

delayed_rescheduled = pd.concat([delayed, rescheduled]).drop_duplicates().reset_index(drop=True)

In [17]:
delayed.to_excel("Results/Delay_reason_onlyDelayed.xlsx", index=False)
rescheduled.to_excel("Results/Delay_reason_onlyRescheduled.xlsx", index=False)

In [18]:
print("Delayed:", delayed.shape)
print("Rescheduled:", rescheduled.shape)

Delayed: (484, 6)
Rescheduled: (317, 6)
